In [1]:
!conda run -p /home/njm12/ATMS_523/envs/xarray-climate python -m ipykernel install --user --name xarray-climate --display-name "Python (xarray-climate)"

Installed kernelspec xarray-climate in /home/njm12/.local/share/jupyter/kernels/xarray-climate



In [1]:
import sys
print(sys.executable)

/home/njm12/ATMS_523/envs/xarray-climate/bin/python


In [2]:
# ================================================================
# Imports
# ================================================================

import pandas as pd
from astral.sun import sun
from astral import LocationInfo
import pytz

In [4]:
# ================================================================
# File paths
# ================================================================

tornado_csv = "/home/njm12/ATMS_596/1950-2024_actual_tornadoes.csv"
output_file = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times_and_EET_bins.csv"

# ================================================================
# Configuration
# ================================================================

states_of_interest = ["MO", "IA", "IL"]
min_date = pd.to_datetime("1950-01-01")

In [5]:
# ================================================================
# Load data
# ================================================================

tornadoes = pd.read_csv(tornado_csv, parse_dates=["date"])

tornadoes = tornadoes[
    tornadoes["st"].isin(states_of_interest) &
    tornadoes["mag"].isin([0,1,2,3,4,5]) &
    (tornadoes["date"] >= min_date)
].copy()

print("Filtered tornadoes:", tornadoes.shape)

# ================================================================
# Fix longitude + create datetime
# ================================================================

tornadoes["slon"] = tornadoes["slon"].apply(lambda x: -x if x > 0 else x)

tornadoes["datetime"] = pd.to_datetime(
    tornadoes["date"].astype(str) + " " + tornadoes["time"].astype(str)
)

central = pytz.timezone("America/Chicago")
tornadoes["datetime"] = tornadoes["datetime"].dt.tz_localize(central)

Filtered tornadoes: (8287, 29)


In [6]:
# ================================================================
# Compute solar times
# ================================================================

def compute_solar_times(row):

    lat = row["slat"]
    lon = -abs(row["slon"])
    date = row["date"]

    try:
        location = LocationInfo(latitude=lat, longitude=lon)
        s = sun(location.observer, date=date, tzinfo=central)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        if sunset < sunrise:
            sunset += pd.Timedelta(days=1)

        return pd.Series({
            "sunrise_local": sunrise,
            "sunset_local": sunset
        })

    except:
        return pd.Series({
            "sunrise_local": pd.NaT,
            "sunset_local": pd.NaT
        })

solar_times = tornadoes.apply(compute_solar_times, axis=1)
df = pd.concat([tornadoes, solar_times], axis=1)

In [8]:
# ================================================================
# Day vs Night classification
# ================================================================

def classify_day_night(row):

    event_time = row["datetime"]
    sunrise = row["sunrise_local"]
    sunset = row["sunset_local"]

    if pd.isnull(sunrise) or pd.isnull(sunset):
        return None

    if sunrise <= event_time <= sunset:
        return "Day"
    else:
        return "Night"

df["day_night_bin"] = df.apply(classify_day_night, axis=1)

In [9]:
# ================================================================
# Repair missing classifications (if any)
# ================================================================

missing = df[df["day_night_bin"].isna()].copy()
print("Rows missing solar classification:", len(missing))

for idx, row in missing.iterrows():

    try:
        lat = row["slat"]
        lon = -abs(row["slon"])
        date = row["date"]

        location = LocationInfo(latitude=lat, longitude=lon)
        s = sun(location.observer, date=date, tzinfo=central)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        if sunset < sunrise:
            sunset += pd.Timedelta(days=1)

        df.loc[idx, "sunrise_local"] = sunrise
        df.loc[idx, "sunset_local"] = sunset

        event_time = df.loc[idx, "datetime"]

        if sunrise <= event_time <= sunset:
            df.loc[idx, "day_night_bin"] = "Day"
        else:
            df.loc[idx, "day_night_bin"] = "Night"

    except Exception as e:
        print("Still failed:", idx, e)

print("Remaining missing:", df["day_night_bin"].isna().sum())


Rows missing solar classification: 0
Remaining missing: 0


In [10]:
# ================================================================
# Refined nocturnal classification (Option A)
# ================================================================

def classify_nocturnal_bins(row):

    event_time = row["datetime"]
    sunrise = row["sunrise_local"]
    sunset = row["sunset_local"]

    if pd.isnull(sunrise) or pd.isnull(sunset):
        return None

    # Day
    if sunrise <= event_time <= sunset:
        return "Day"

    # Define key windows
    eet_start = sunset - pd.Timedelta(hours=2)
    eet_end   = sunset + pd.Timedelta(hours=2)

    post_start = sunset + pd.Timedelta(hours=2)
    post_end   = sunset + pd.Timedelta(hours=6)

    # Early Evening Transition (EET)
    if eet_start <= event_time <= eet_end:
        return "Early Evening Transition"

    # Post-transition nocturnal period (LLJ strengthening window)
    elif post_start < event_time <= post_end:
        return "Post-Transition"

    # Core night
    else:
        return "Core Night"

df["nocturnal_bin"] = df.apply(classify_nocturnal_bins, axis=1)

In [11]:
# ================================================================
# Diagnostics
# ================================================================

print("\nDay/Night counts:")
print(df["day_night_bin"].value_counts())

print("\nRefined nocturnal bin counts:")
print(df["nocturnal_bin"].value_counts())

# ================================================================
# Save output
# ================================================================

df.to_csv(output_file, index=False)

print("\nSaved to:", output_file)


Day/Night counts:
day_night_bin
Day      6155
Night    2132
Name: count, dtype: int64

Refined nocturnal bin counts:
nocturnal_bin
Day                         6155
Early Evening Transition     933
Post-Transition              613
Core Night                   586
Name: count, dtype: int64

Saved to: /home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times_and_EET_bins.csv
